# Action Predictor Training: TensorFlow

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization, Activation # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.callbacks import EarlyStopping # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore
import argparse
import os
import joblib
import warnings
import gc # Garbage collector

# --- Suppress TensorFlow/warnings ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

## 2. Configuration

Instead of command-line arguments, set the training parameters in this cell.

In [2]:
# --- Main Settings ---
PARQUET_PATH = "10k.parquet" 
FEATURE_SET = 'medium'      
PREDICT_MODE = 'move_only'  

# --- Data Splitting ---
MIN_TURN = 1
TEST_SPLIT_SIZE = 0.2
VAL_SPLIT_SIZE = 0.15

# --- TensorFlow Model Hyperparameters ---
EPOCHS = 30
BATCH_SIZE = 2000
LEARNING_RATE = 0.001

## 3. Helper & Training Functions

These are the core functions for finding active Pokémon and for defining and training the TensorFlow model.

In [3]:
def find_active_species(row, player_prefix):
    """
    Finds the species of the active Pokemon for a given player prefix ('p1' or 'p2')
    in a DataFrame row containing slot information.
    """
    for i in range(1, 7): # Check slots 1 to 6
        active_col = f"{player_prefix}_slot{i}_is_active"
        species_col = f"{player_prefix}_slot{i}_species"
        if active_col in row.index and species_col in row.index:
            if row[active_col] == 1:
                return row[species_col] if pd.notna(row[species_col]) else 'Unknown'
    return 'Unknown' 

def train_tensorflow_action_predictor(X_train_processed, X_val_processed, X_test_processed,
                                      y_train_encoded, y_val_encoded, y_test_encoded, # INTEGER encoded y
                                      num_classes, class_weight_dict, label_encoder, # Pass num_classes, weights, encoder
                                      epochs=20, batch_size=128, learning_rate=0.001,
                                      label_suffix=""): 
    print(f"\n--- Training TensorFlow Model ---")
    print(f"Input shape: {X_train_processed.shape[1]}")
    print(f"Num classes: {num_classes}")

    # Convert integer labels to one-hot encoding
    print("One-hot encoding target variable for TF...")
    try:
        y_train_one_hot = to_categorical(y_train_encoded, num_classes=num_classes)
        y_val_one_hot = to_categorical(y_val_encoded, num_classes=num_classes)
        y_test_one_hot = to_categorical(y_test_encoded, num_classes=num_classes)
        print("One-hot encoding complete.")
    except (ValueError, MemoryError) as e:
         print(f"Error during one-hot encoding: {e}")
         return None, None

    # Define the Model
    input_dim = X_train_processed.shape[1]
    print(f"Building TF model with input dimension: {input_dim}")
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(256, use_bias=False),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.3),
        Dense(128, use_bias=False),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.3),
        Dense(64, use_bias=False),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')])
    model.summary()

    # Train the Model
    print("\nStarting TF model training...")
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
    history = model.fit(
        X_train_processed, y_train_one_hot,
        validation_data=(X_val_processed, y_val_one_hot),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=[early_stopping],
        verbose=2
    )
    print("TF Training finished.")

    # Evaluate the Model
    print("\nEvaluating TF model on the test set...")
    results = model.evaluate(X_test_processed, y_test_one_hot, verbose=0)
    loss, accuracy, top_5_accuracy = results[0], results[1], results[2] if len(results) > 2 else np.nan
    print(f"TF Test Loss: {loss:.4f}")
    print(f"TF Test Accuracy: {accuracy:.4f}")
    print(f"TF Test Top-5 Accuracy: {top_5_accuracy:.4f}")

    y_pred_proba = model.predict(X_test_processed, verbose=0)

    y_pred_indices = np.argmax(y_pred_proba, axis=1)

    y_test_indices = np.argmax(y_test_one_hot, axis=1)

    # 4. Calculate the weighted F1 score
    weighted_f1 = f1_score(y_test_indices, y_pred_indices, average='weighted', zero_division=0)
    print(f"TF Test F1 Accuracy: {weighted_f1:.4f}")

    # Save Model
    model_save_path = f'action_tf_model_v4_{label_suffix}.keras'
    print(f"Saving TF model to {model_save_path}")
    model.save(model_save_path)
    print("TF Model saved.")
    return history, model

## 4. Data Loading and Filtering

In [4]:
print(f"--- Starting Action Predictor Training ---")
print(f"Model type: TENSORFLOW")
print(f"Feature Set: {FEATURE_SET.upper()}")
print(f"Prediction Mode: {PREDICT_MODE.upper()}")
print(f"Loading data from: {PARQUET_PATH}")

try:
    df = pd.read_parquet(PARQUET_PATH)
    print(f"Original data shape: {df.shape}")
except Exception as e:
    print(f"Error loading Parquet file: {e}")
    # In a notebook, we might stop here or raise the error
    raise e

# --- Filter Data ---
print("\nFiltering data...")
df = df.dropna(subset=['action_taken'])
print(f"Rows after dropping NaN action_taken: {len(df)}")

print(f"Filtering for player_to_move == 'p1' (for {FEATURE_SET} set)...")
df = df[df['player_to_move'] == 'p1'].copy()
print(f"Rows after filtering for p1's move: {len(df)}")


# Filter based on predict_mode to define the target variable y_raw
if PREDICT_MODE == 'move_only':
    print(f"\nFiltering for actions starting with 'move:'...")
    df = df[df['action_taken'].astype(str).str.startswith('move:')].copy()
    if df.empty:
        raise ValueError("Error: No 'move:' actions found. Cannot train in 'move_only' mode.")
    print(f"Rows after filtering for move actions: {len(df)}")
    y_raw = df['action_taken'].str.replace('move:', '', regex=False)
    print("Target variable 'y_raw' now contains only move names.")
elif PREDICT_MODE == 'all_actions':
    print("\nUsing all actions (moves and switches) as target.")
    y_raw = df['action_taken'].copy()
else:
    raise ValueError(f"Error: Invalid PREDICT_MODE '{PREDICT_MODE}'.")

if MIN_TURN > 0:
    df = df[df['turn_number'] >= MIN_TURN].copy()
    print(f"Rows after filtering turns >= {MIN_TURN}: {len(df)}")
    if df.empty:
        raise ValueError("No data remaining after turn filtering.")

--- Starting Action Predictor Training ---
Model type: TENSORFLOW
Feature Set: MEDIUM
Prediction Mode: MOVE_ONLY
Loading data from: 10k.parquet
Original data shape: (513416, 184)

Filtering data...
Rows after dropping NaN action_taken: 513416
Filtering for player_to_move == 'p1' (for medium set)...
Rows after filtering for p1's move: 256619

Filtering for actions starting with 'move:'...
Rows after filtering for move actions: 166742
Target variable 'y_raw' now contains only move names.
Rows after filtering turns >= 1: 166742


## 5. Feature Engineering

In [ ]:
X = None
numerical_features = []
categorical_features = []


print("\n--- Using MEDIUM feature set (with Active Revealed Moves) ---")
selected_columns = []
base_active_features = ['species', 'hp_perc', 'status', 'boost_atk', 'boost_def', 'boost_spa', 'boost_spd', 'boost_spe','terastallized', 'tera_type']
bench_cols = [f'{p}_slot{i}_{f}' for i in range(1, 7) for p in ['p1', 'p2'] for f in ['hp_perc', 'status', 'species']]
selected_columns.extend(bench_cols)
field_cols = ['field_weather', 'field_terrain', 'field_pseudo_weather']
selected_columns.extend(field_cols)
hazard_cols = [f'{p}_hazard_{h.replace(" ", "_")}' for p in ['p1', 'p2'] for h in ['stealth_rock', 'spikes', 'toxic_spikes', 'sticky_web']]
selected_columns.extend(hazard_cols)
side_cond_cols = [f'{p}_side_{c.lower().replace(" ", "_")}' for p in ['p1', 'p2'] for c in ['reflect', 'light_screen', 'aurora_veil', 'tailwind']]
selected_columns.extend(side_cond_cols)
context_cols = ['last_move_p1', 'last_move_p2', 'turn_number']
selected_columns.extend(context_cols)
valid_selected_columns = [col for col in selected_columns if col in df.columns]
X_medium = df[valid_selected_columns].copy()
active_data = {}
for idx, row in df.iterrows():
    active_p1_slot, active_p2_slot = -1, -1
    for i_slot in range(1, 7):
        if row.get(f'p1_slot{i_slot}_is_active', 0) == 1: active_p1_slot = i_slot
        if row.get(f'p2_slot{i_slot}_is_active', 0) == 1: active_p2_slot = i_slot
    row_active_data = {}
    p1_active_moves_str = 'none'
    if active_p1_slot != -1:
        for feat in base_active_features: row_active_data[f'p1_active_{feat}'] = row.get(f'p1_slot{active_p1_slot}_{feat}', None)
        p1_active_moves_str = row.get(f'p1_slot{active_p1_slot}_revealed_moves', 'none')
    else:
        for feat in base_active_features: row_active_data[f'p1_active_{feat}'] = None
    row_active_data['p1_active_revealed_moves_str'] = p1_active_moves_str
    p2_active_moves_str = 'none'
    if active_p2_slot != -1:
        for feat in base_active_features: row_active_data[f'p2_active_{feat}'] = row.get(f'p2_slot{active_p2_slot}_{feat}', None)
        p2_active_moves_str = row.get(f'p2_slot{active_p2_slot}_revealed_moves', 'none')
    else:
        for feat in base_active_features: row_active_data[f'p2_active_{feat}'] = None
    row_active_data['p2_active_revealed_moves_str'] = p2_active_moves_str
    active_data[idx] = row_active_data
active_df = pd.DataFrame.from_dict(active_data, orient='index')
X = pd.concat([X_medium, active_df], axis=1)
del X_medium, active_df, active_data; gc.collect()
active_revealed_move_cols = ['p1_active_revealed_moves_str', 'p2_active_revealed_moves_str']
active_all_revealed_moves = set()
for col in active_revealed_move_cols:
    if col in X.columns: active_all_revealed_moves.update(m for m in X[col].fillna('none').astype(str).str.split(',').explode().unique() if m and m != 'none')
unique_moves_list = sorted(list(active_all_revealed_moves))
new_binary_move_cols = []
for base_col in active_revealed_move_cols:
    if base_col in X.columns:
        player_prefix = base_col.split('_')[0]
        revealed_sets = X[base_col].fillna('none').astype(str).str.split(',').apply(set)
        for move in unique_moves_list:
            new_col_name = f"{player_prefix}_active_revealed_move_{move.replace(' ', '_').replace('-', '_')}"
            X[new_col_name] = revealed_sets.apply(lambda s: 1 if move in s else 0).astype(np.int8)
            new_binary_move_cols.append(new_col_name)
X = X.drop(columns=[col for col in active_revealed_move_cols if col in X.columns])
for p in ['p1', 'p2']:
    if f'{p}_active_species' in X.columns: categorical_features.append(f'{p}_active_species')
    if f'{p}_active_status' in X.columns: categorical_features.append(f'{p}_active_status')
    # ... (rest of the feature type identification from your script)
numerical_features.extend(new_binary_move_cols)
numerical_features = sorted(list(set(numerical_features)))
categorical_features = sorted(list(set(categorical_features)))

    
# --- Final NaN Cleanup ---
print("\nFinal NaN Check...")
for col in numerical_features:
    if col in X.columns and X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())
for col in categorical_features:
    if col in X.columns and X[col].isnull().any():
        X[col] = X[col].fillna('Unknown').astype('category')

print(f"Final feature counts: Numerical={len(numerical_features)}, Categorical={len(categorical_features)}")
print(f"Final X shape: {X.shape}")


--- Using MEDIUM feature set (with Active Revealed Moves) ---

Final NaN Check...
Final feature counts: Numerical=1104, Categorical=4
Final X shape: (166742, 1168)


## 6. Target Encoding and Data Splitting

In [ ]:
# --- Encode Target Variable (y) ---
print("\nEncoding target variable...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw.astype(str))
num_classes = len(label_encoder.classes_)
print(f"Found {num_classes} unique actions/moves.")

label_encoder_suffix = f"{FEATURE_SET}_{PREDICT_MODE}"
label_encoder_path = f'action_label_encoder_v4_{label_encoder_suffix}.joblib'
joblib.dump(label_encoder, label_encoder_path)
print(f"Label encoder saved to {label_encoder_path}")
del y_raw; gc.collect()

# --- Split Data ---
print("\nSplitting data into Train, Validation, Test sets...")
X_train_full, X_test, y_train_full_encoded, y_test_encoded = train_test_split(
    X, y_encoded, test_size=TEST_SPLIT_SIZE, random_state=42
)
val_size_relative = VAL_SPLIT_SIZE / (1.0 - TEST_SPLIT_SIZE)
X_train, X_val, y_train_encoded, y_val_encoded = train_test_split(
    X_train_full, y_train_full_encoded, test_size=val_size_relative, random_state=42
)

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")
del X, df, X_train_full, y_train_full_encoded; gc.collect()


Encoding target variable...
Found 515 unique actions/moves.
Label encoder saved to action_label_encoder_v4_medium_move_only.joblib

Splitting data into Train, Validation, Test sets...
Train shape: (108381, 1168)
Val shape: (25012, 1168)
Test shape: (33349, 1168)


0

## 7. Class Weights and Preprocessing

In [7]:
# --- Calculate Class Weights ---
print("\nCalculating class weights...")
unique_classes, class_counts = np.unique(y_train_encoded, return_counts=True)
class_weights_values = compute_class_weight('balanced', classes=unique_classes, y=y_train_encoded)
class_weight_dict = dict(zip(unique_classes, class_weights_values))
for i in range(num_classes):
    if i not in class_weight_dict:
        class_weight_dict[i] = 1.0
print("Class weights calculated.")

# --- Preprocess X data for TensorFlow ---
print("\nSetting up TF preprocessing pipeline (OneHotEncoder + Scaler)...")

# Ensure feature lists only contain columns present in the final X_train
final_train_cols = X_train.columns.tolist()
numerical_features = [f for f in numerical_features if f in final_train_cols]
categorical_features = [f for f in categorical_features if f in final_train_cols]

transformers = []
if numerical_features:
    transformers.append(('num', StandardScaler(), numerical_features))
if categorical_features:
    transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

print("Applying TF preprocessing (fit on train, transform all)...")
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed shapes - Train: {X_train_processed.shape}, Val: {X_val_processed.shape}, Test: {X_test_processed.shape}")

# Save the preprocessor
preprocessor_path = f'action_tf_preprocessor_v4_{label_encoder_suffix}.joblib'
joblib.dump(preprocessor, preprocessor_path)
print(f"TF preprocessor saved to {preprocessor_path}")

del X_train, X_val, X_test; gc.collect()


Calculating class weights...
Class weights calculated.

Setting up TF preprocessing pipeline (OneHotEncoder + Scaler)...
Applying TF preprocessing (fit on train, transform all)...
Processed shapes - Train: (108381, 2249), Val: (25012, 2249), Test: (33349, 2249)
TF preprocessor saved to action_tf_preprocessor_v4_medium_move_only.joblib


22

## 8. Train the Model

Finally, we pass the preprocessed data to our training function to build and train the neural network.

In [8]:
if X_train_processed is not None:
    train_tensorflow_action_predictor(
        X_train_processed, X_val_processed, X_test_processed,
        y_train_encoded, y_val_encoded, y_test_encoded,
        num_classes, class_weight_dict, label_encoder,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        label_suffix=label_encoder_suffix
    )
else:
    print("Skipping TF training due to preprocessing errors.")


--- Training TensorFlow Model ---
Input shape: 2249
Num classes: 515
One-hot encoding target variable for TF...
One-hot encoding complete.
Building TF model with input dimension: 2249


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       575,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 515)            │        33,475 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 651,971 (2.49 MB)

 Trainable params: 651,075 (2.48 MB)

 Non-trainable params: 896 (3.50 KB)


Starting TF model training...
Epoch 1/30
55/55 - 7s - 120ms/step - accuracy: 0.0038 - loss: 6.4387 - top_5_accuracy: 0.0179 - val_accuracy: 0.0249 - val_loss: 6.1754 - val_top_5_accuracy: 0.0717
Epoch 2/30
55/55 - 3s - 56ms/step - accuracy: 0.0215 - loss: 6.0687 - top_5_accuracy: 0.0676 - val_accuracy: 0.0900 - val_loss: 6.0593 - val_top_5_accuracy: 0.1689
Epoch 3/30
55/55 - 3s - 59ms/step - accuracy: 0.0513 - loss: 5.7980 - top_5_accuracy: 0.1244 - val_accuracy: 0.1421 - val_loss: 5.8357 - val_top_5_accuracy: 0.2554
Epoch 4/30
55/55 - 3s - 55ms/step - accuracy: 0.0770 - loss: 5.5513 - top_5_accuracy: 0.1696 - val_accuracy: 0.1640 - val_loss: 5.6121 - val_top_5_accuracy: 0.2939
Epoch 5/30
55/55 - 3s - 63ms/step - accuracy: 0.0951 - loss: 5.3119 - top_5_accuracy: 0.2023 - val_accuracy: 0.1810 - val_loss: 5.4418 - val_top_5_accuracy: 0.3347
Epoch 6/30
55/55 - 3s - 54ms/step - accuracy: 0.1135 - loss: 5.0414 - top_5_accuracy: 0.2374 - val_accuracy: 0.1901 - val_loss: 5.2789 - val_top_5_a